In [1]:
%%writefile rag_engine.py
import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  
import gc  
import openpyxl  
import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
import json
import re
import re
import json

Overwriting rag_engine.py


In [2]:
%%writefile -a rag_engine.py

def find_parent_large_chunks(small_docs, large_docs):
    """البحث عن القطع الكبيرة التي تحتوي على نص القطع الصغيرة المسترجعة"""
    parent_chunks = []
    unique_content_set = set()
    for small_doc in small_docs:
        for large_doc in large_docs:
            if small_doc.page_content in large_doc.page_content:
                if large_doc.page_content not in unique_content_set:
                    parent_chunks.append(large_doc)
                    unique_content_set.add(large_doc.page_content)
                break
    return parent_chunks

Appending to rag_engine.py


In [3]:
%%writefile -a rag_engine.py
import os

def initialize_fixed_knowledge(processor, embed_model):
    fixed_files = ["modified_comp_updated3 (1).txt", "Depot_Updated.txt", "thirdcon.txt", 
                   "RAG_STAGE6_cleaned.txt", "private_cond.txt", "resident_cleaned.txt", "email_history.txt"]
    
    initial_docs = []
    for f in fixed_files:
        if os.path.exists(f):
            text = processor.process_text(f)
            initial_docs.append(Document(page_content=text, metadata={'source': f}))

    pattern_split_docs = regex_split_documents(initial_docs)
    text_splitter_large = RecursiveCharacterTextSplitter(chunk_size=1300, chunk_overlap=200)
    text_splitter_small = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=25)
    
    docs_large = []
    docs_small = []
    for doc in pattern_split_docs:
        heading_match = re.match(r"^(اللائحة المادة\s+\d+|النظام المادة\s+\d+|المادة\s+\d+|محضر تسليم|الموقع:|الفقرة\s+\d+|إيميل وارد بتاريخ\s+.*|إيميل صادر بتاريخ\s+.*)", doc.page_content)
        heading = heading_match.group(0) if heading_match else ""
        
        for chunk in text_splitter_large.split_text(doc.page_content):
            docs_large.append(Document(page_content=f"{heading} {chunk}", metadata=doc.metadata))
        
        for chunk in text_splitter_small.split_text(doc.page_content):
            docs_small.append(Document(page_content=f"{heading} {chunk}", metadata=doc.metadata))

    vs_large = FAISS.from_documents(docs_large, embed_model)
    vs_small = FAISS.from_documents(docs_small, embed_model)

    bm25_large = BM25Retriever.from_documents(docs_large)
    bm25_large.k = 6  
    
    bm25_small = BM25Retriever.from_documents(docs_small)
    bm25_small.k = 18
    
    print(f"✅ تم بناء القاعدة المعرفية: {len(docs_large)} قطع كبيرة و {len(docs_small)} قطع صغيرة.")
    
    return {
        "vs_large": vs_large,
        "vs_small": vs_small,
        "bm25_large": bm25_large,
        "bm25_small": bm25_small,
        "docs_large": docs_large
    }



Appending to rag_engine.py


In [4]:
%%writefile -a rag_engine.py

def regex_split_documents(documents):
    """تقسيم النصوص بناءً على أنماط المواد واللوائح لضمان عدم انكسار النص القانوني"""
    pattern = r"(?=(اللائحة المادة\s+\d+\s+من\s+نظام\s+المنافسات\s+والمشتريات\s+الحكومية|النظام المادة\s+\d+\s+من\s+نظام\s+المنافسات\s+والمشتريات\s+الحكومية|المادة\s+\d+\s+من\s+قواعد\s+واجراءات\s+المستودعات\s+الحكومية|محضر تسليم(?:\s+\S+){1,6}|الموقع:|الفقرة\s+\d+|إيميل وارد بتاريخ\s+[\d-]+\s+[\d:]+|إيميل صادر بتاريخ\s+[\d-]+\s+[\d:]+))"
    
    new_documents = []
    for doc in documents:
        content = doc.page_content
        metadata = doc.metadata
        parts = re.split(pattern, content)
        
        current_chunk = ""
        current_heading = ""
        for part in parts:
            if not part or not part.strip(): continue
            part = part.strip()
            
            # التحقق إذا كان الجزء هو عنوان مادة (بشكل صارم)
            is_heading = re.match(r"^(اللائحة المادة\s+\d+|النظام المادة\s+\d+|المادة\s+\d+|محضر تسليم|الموقع:|الفقرة\s+\d+|إيميل وارد بتاريخ\s+.*|إيميل صادر بتاريخ\s+.*)", part)
            
            if is_heading:
                if current_chunk.strip():
                    new_documents.append(Document(page_content=current_chunk.strip(), metadata=metadata))
                current_heading = part
                current_chunk = f"{current_heading} "  # تكرار العنوان داخل المحتوى كما في المرفق
            elif current_chunk:
                current_chunk += part + " "
                
        if current_chunk.strip():
            new_documents.append(Document(page_content=current_chunk.strip(), metadata=metadata))
            
    return new_documents


Appending to rag_engine.py


In [5]:
%%writefile -a rag_engine.py

def get_fixed_context(query, kb):
    """استرجاع هجين مكثف: 8 كبيرة و 16 صغيرة من FAISS و BM25 وتحويلها للأصول"""
    
    # 1. الاسترجاع المباشر للقطع الكبيرة (K=8 من كل مصدر)
    faiss_large_docs = kb["vs_large"].as_retriever(search_kwargs={"k": 6}).invoke(query) # تم التحديث إلى 8
    bm25_large_docs = kb["bm25_large"].invoke(query) # ستسترجع 8 بناءً على التهيئة السابقة
    
    # 2. الاسترجاع للقطع الصغيرة (K=16 من كل مصدر)
    faiss_small_docs = kb["vs_small"].as_retriever(search_kwargs={"k": 18}).invoke(query) # تم التأكيد على 16
    bm25_small_docs = kb["bm25_small"].invoke(query) # ستسترجع 16 بناءً على التهيئة السابقة
    
    # دمج نتائج القطع الصغيرة للبحث عن أصولها
    all_small_hits = faiss_small_docs + bm25_small_docs
    
    # 3. تحويل كافة القطع الصغيرة المسترجعة (32 قطعة كحد أقصى) إلى أصولها الكبيرة (Parents)
    parents_from_small = find_parent_large_chunks(all_small_hits, kb["docs_large"])
    
    # 4. دمج كل النتائج (8 كبيرة FAISS + 8 كبيرة BM25 + أصول القطع الصغيرة) مع منع التكرار
    all_context_docs = faiss_large_docs + bm25_large_docs + parents_from_small
    
    unique_content = set()
    final_context_list = []
    
    for doc in all_context_docs:
        if doc.page_content not in unique_content:
            final_context_list.append(doc.page_content)
            unique_content.add(doc.page_content)
            
    return "\n---\n".join(final_context_list)

Appending to rag_engine.py


In [6]:
%%writefile -a rag_engine.py

def get_attachment_context(attachments_list, query, processor, embed_model):
    """
    يعالج المرفقات بتقسيمها لقطع (كبيرة وصغيرة)، يبحث في الصغيرة ويسترجع الكبيرة المقابلة لها.
    """
    if not attachments_list: return ""
    
    docs_large = [] 
    docs_small = [] 
    mandatory_headers = [] 
    
    text_splitter_large = RecursiveCharacterTextSplitter(chunk_size=1400, chunk_overlap=200)
    text_splitter_small = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=25)
    
    for path in attachments_list:
        try:
            text = ""
            if path.endswith('.pdf'): text = str(processor.process_pdf(path))
            elif path.endswith(('.xlsx', '.xls')): text = str(processor.process_excel(path))
            elif path.endswith('.docx'): text = processor.process_docx(path)
            elif path.endswith(('.txt', '.note')): text = processor.process_text(path)
            else: continue
            
            if not text or not text.strip(): continue
            
            file_name = os.path.basename(path)
            
            mandatory_headers.append(f"--- مقدمة ملف: {file_name} ---\n{text[:1000]}...")
            
            file_large_chunks = []
            for chunk in text_splitter_large.split_text(text):
                file_large_chunks.append(Document(page_content=chunk, metadata={'source': file_name}))
            docs_large.extend(file_large_chunks)
            
            file_small_chunks = []
            for chunk in text_splitter_small.split_text(text):
                file_small_chunks.append(Document(page_content=chunk, metadata={'source': file_name}))
            docs_small.extend(file_small_chunks)
            
            del text
            del file_large_chunks
            del file_small_chunks
            gc.collect()

        except Exception as e:
            print(f"⚠ خطأ في معالجة المرفق {path}: {e}")
            gc.collect()

    if not docs_small: return ""
    
    vs_small_attach = FAISS.from_documents(docs_small, embed_model)
    
    small_results = vs_small_attach.as_retriever(search_kwargs={"k": 18}).invoke(query)
    

    parent_results = find_parent_large_chunks(small_results, docs_large)
    
    final_attachment_ctx = []
    final_attachment_ctx.append("### [معلومات أساسية من المرفقات]:")
    final_attachment_ctx.extend(mandatory_headers)
    
    final_attachment_ctx.append("\n### [تفاصيل موسعة مسترجعة من المرفقات]:")
    unique_content = set()
    for doc in parent_results:
        content = f"[{doc.metadata['source']}]: {doc.page_content}"
        if content not in unique_content:
            final_attachment_ctx.append(content)
            unique_content.add(content)
    
    # تنظيف نهائي للكائنات الكبيرة
    del docs_large
    del docs_small
    del vs_small_attach
    gc.collect()
    
    return "\n---\n".join(final_attachment_ctx)


Appending to rag_engine.py
